In [1]:
import sys
import os

# 경로 설정: 스크립트 경로에서 상위 디렉토리로 이동한 후 src 경로 추가
health_data_path = os.path.abspath(os.path.join('..', 'src'))
health_learning_data_path = os.path.abspath(os.path.join(os.getcwd(), "../../health_learning_data/health_data/src"))
preprocessing_path = os.path.abspath(os.path.join(os.getcwd(), "../../preprocessing/src"))

paths = [health_data_path, health_learning_data_path, preprocessing_path]

def add_paths(paths):
    """
    지정된 경로들이 sys.path에 없으면 추가하는 함수.
    
    Parameters:
    - paths (list): 추가하려는 경로들의 리스트.
    """
    for path in paths:
        if path not in sys.path:
            sys.path.append(path)
            print(f"Path added: {path}")
        else:
            print(f"Path already exists: {path}")
            
add_paths(paths)

Path added: C:\Users\pc021\Desktop\dx_project\techross\health_data\src
Path added: C:\Users\pc021\Desktop\dx_project\techross\health_learning_data\health_data\src
Path added: C:\Users\pc021\Desktop\dx_project\techross\preprocessing\src


In [2]:
# basic
import pandas as pd
import schedule
import time

# module.algorithms
from my_package.algorithms.total_system_health_algorithm import apply_system_health_algorithms_with_total
from update_package.algorithms.total_system_health_learning_algorithm import apply_system_health_learning_algorithms_with_total
from preprocessing.load_processing import distribute_by_application

# module.data
from my_package.data.scheduled_data_fetcher import fetch_data_on_schedule
from my_package.data.logger_confg import logger

Path already exists: C:\Users\pc021\Desktop\dx_project\techross\health_data\src
Path already exists: C:\Users\pc021\Desktop\dx_project\techross\health_learning_data\health_data\src
Path already exists: C:\Users\pc021\Desktop\dx_project\techross\preprocessing\src


In [3]:
def get_latest_date_on_schedule():
    
    fetched_data = fetch_data_on_schedule('ecs_latest_data')

    grouped_data = fetched_data.groupby(['SHIP_ID','OP_INDEX','SECTION']).count() 
    
    grouped_index = grouped_data.index

    return grouped_index, fetched_data

In [4]:
# ship_id = 'T20191002002'
# grouped_index = get_latest_date_on_schedule(data, ship_id)

# datas = pd.DataFrame(index = grouped_index).reset_index()
# datas[(datas['SHIP_ID']=='T170713-01401') & (datas['OP_INDEX']==1685) & (datas['SECTION']==0)]

In [5]:
def schedule_health_assessment():
    
    grouped_index, fetched_data = get_latest_date_on_schedule()
    
    for index in grouped_index:
        ship_id = index[0]
        op_index = index[1]
        section = index[2] 
        
        # 데이터 처리를 위한 갯수 조건을 만족하는지 판단
        selected_df = fetched_data[(fetched_data['SHIP_ID']==ship_id) & (fetched_data['OP_INDEX']==op_index) & (fetched_data['SECTION']==section)]
        date_time = selected_df.iloc[0]['DATA_TIME']
        data_len = len(selected_df)
        
        print(f'SHIP_ID : {ship_id} / OP_INDEX : {op_index} / SECTION : {section} -  데이터 선택 ({data_len})')
        
        if (data_len>=160) :
            print(f'SHIP_ID : {ship_id} / OP_INDEX : {op_index} / SECTION : {section} -  조건 통과')         
            try:
                sensor, preprocessed = distribute_by_application(ship_id=ship_id, op_index=op_index, section=section)
                if sensor is None and preprocessed is None:
                    print("선박 데이터 프레임이 존재하지 않습니다.")
                    continue       

                elif preprocessed is not None:
                    logger.info(f'SHIP_ID={ship_id} | OP_INDEX={op_index} | SECTION={section} | START_TIME={date_time} | LOG_ENTRY=모델 및 통계 패키지 통해 결과를 도출했습니다 | TYPE=ALL | is_processed=True')
                    print("전처리 후 학습 데이터 프레임이 존재합니다.")
                    pass
                    # apply_system_health_algorithms_with_total(sensor, ship_id, op_index, section)
                    # apply_system_health_learning_algorithms_with_total(data=preprocessed, ship_id=ship_id, op_index=op_index, section=section)
                else:
                    logger.info(f'SHIP_ID={ship_id} | OP_INDEX={op_index} | SECTION={section} | START_TIME={date_time}  | LOG_ENTRY=전처리 후 모델 데이터 프레임이 존재하지 않아 통계 알고리즘만 단독으로 진행합니다. | TYPE=STATS | is_processed=True')
                    print("전처리 후 모델 데이터 프레임이 존재하지 않아 통계 알고리즘 단독 진행합니다.")
                    pass
                #     apply_system_health_algorithms_with_total(data=sensor, ship_id=ship_id, op_index=op_index, section=section)
                
            except ValueError as e :
                print(f'에러 발생: {e}. 다음 반복으로 넘어갑니다.')
                continue  # 에러 발생 시 다음 반복으로 넘어감\

            except KeyError as e :
                print(f'에러 발생: {e}. 다음 반복으로 넘어갑니다.')

            except TypeError as e :
                print(f'에러 발생: {e}. 다음 반복으로 넘어갑니다.')
                continue  # 에러 발생 시 다음 반복으로 넘어감
                
            except IndexError as e :
                print(f'에러 발생: {e}. 다음 반복으로 넘어갑니다.')
                continue  # 에러 발생 시 다음 반복으로 넘어감 
        else:
            logger.info(f'SHIP_ID={ship_id} | OP_INDEX={op_index} | SECTION={section} | START_TIME={date_time} | LOG_ENTRY=데이터 개수가 {data_len}여서 조건을 만족하지 않습니다. | TYPE=MAIN | is_processed=False')

In [6]:
schedule_health_assessment()

SHIP_ID : T130422-00101 / OP_INDEX : 59 / SECTION : 0 -  데이터 선택 (8)
SHIP_ID : T130422-00101 / OP_INDEX : 60 / SECTION : 0 -  데이터 선택 (14)
SHIP_ID : T130422-00101 / OP_INDEX : 61 / SECTION : 0 -  데이터 선택 (31)
SHIP_ID : T140113-00101 / OP_INDEX : 1896 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T140113-00101 / OP_INDEX : 1897 / SECTION : 0 -  데이터 선택 (211)
SHIP_ID : T140113-00101 / OP_INDEX : 1897 / SECTION : 0 -  조건 통과
optype : 2
distribute_by_application 함수 실행 시간: 8.6430초
선박 데이터 프레임이 존재하지 않습니다.
SHIP_ID : T140218-00102 / OP_INDEX : 316 / SECTION : 0 -  데이터 선택 (34)
SHIP_ID : T140218-00102 / OP_INDEX : 317 / SECTION : 0 -  데이터 선택 (34)
SHIP_ID : T140218-00102 / OP_INDEX : 318 / SECTION : 0 -  데이터 선택 (93)
SHIP_ID : T140218-00102 / OP_INDEX : 319 / SECTION : 0 -  데이터 선택 (105)
SHIP_ID : T140218-00102 / OP_INDEX : 320 / SECTION : 0 -  데이터 선택 (31)
SHIP_ID : T140218-00102 / OP_INDEX : 321 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T140218-00102 / OP_INDEX : 322 / SECTION : 0 -  데이터 선택 (11)
SHIP_ID : T140218-0

optype : 2
distribute_by_application 함수 실행 시간: 6.2705초
선박 데이터 프레임이 존재하지 않습니다.
SHIP_ID : T170524-00102 / OP_INDEX : 90 / SECTION : 0 -  데이터 선택 (96)
SHIP_ID : T170524-00102 / OP_INDEX : 91 / SECTION : 0 -  데이터 선택 (205)
SHIP_ID : T170524-00102 / OP_INDEX : 91 / SECTION : 0 -  조건 통과
optype : 2
distribute_by_application 함수 실행 시간: 3.1315초
선박 데이터 프레임이 존재하지 않습니다.
SHIP_ID : T170524-00102 / OP_INDEX : 96 / SECTION : 0 -  데이터 선택 (68)
SHIP_ID : T170524-00103 / OP_INDEX : 104 / SECTION : 0 -  데이터 선택 (59)
SHIP_ID : T170524-00103 / OP_INDEX : 105 / SECTION : 0 -  데이터 선택 (46)
SHIP_ID : T170524-00103 / OP_INDEX : 106 / SECTION : 0 -  데이터 선택 (108)
SHIP_ID : T170524-00103 / OP_INDEX : 107 / SECTION : 0 -  데이터 선택 (92)
SHIP_ID : T170524-00103 / OP_INDEX : 108 / SECTION : 0 -  데이터 선택 (75)
SHIP_ID : T170524-00103 / OP_INDEX : 109 / SECTION : 0 -  데이터 선택 (15)
SHIP_ID : T170524-00103 / OP_INDEX : 110 / SECTION : 0 -  데이터 선택 (147)
SHIP_ID : T170524-00103 / OP_INDEX : 111 / SECTION : 0 -  데이터 선택 (173)
SHIP_ID : 

optype : 2
distribute_by_application 함수 실행 시간: 3.1635초
선박 데이터 프레임이 존재하지 않습니다.
SHIP_ID : T20180817003 / OP_INDEX : 436 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20180817003 / OP_INDEX : 437 / SECTION : 0 -  데이터 선택 (835)
SHIP_ID : T20180817003 / OP_INDEX : 437 / SECTION : 0 -  조건 통과
optype : 2
distribute_by_application 함수 실행 시간: 7.8107초
선박 데이터 프레임이 존재하지 않습니다.
SHIP_ID : T20180817003 / OP_INDEX : 438 / SECTION : 0 -  데이터 선택 (41)
SHIP_ID : T20180817003 / OP_INDEX : 439 / SECTION : 0 -  데이터 선택 (155)
SHIP_ID : T20180817003 / OP_INDEX : 440 / SECTION : 0 -  데이터 선택 (68)
SHIP_ID : T20180817003 / OP_INDEX : 441 / SECTION : 0 -  데이터 선택 (7)
SHIP_ID : T20180910003 / OP_INDEX : 297 / SECTION : 0 -  데이터 선택 (347)
SHIP_ID : T20180910003 / OP_INDEX : 297 / SECTION : 0 -  조건 통과
optype : 2
distribute_by_application 함수 실행 시간: 7.8035초
선박 데이터 프레임이 존재하지 않습니다.
SHIP_ID : T20180910003 / OP_INDEX : 298 / SECTION : 0 -  데이터 선택 (9)
SHIP_ID : T20180910003 / OP_INDEX : 299 / SECTION : 0 -  데이터 선택 (416)
SHIP_ID : T2018091

SHIP_ID : T20181012010 / OP_INDEX : 88 / SECTION : 0 -  데이터 선택 (7)
SHIP_ID : T20181012010 / OP_INDEX : 89 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20181012010 / OP_INDEX : 90 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20181012010 / OP_INDEX : 91 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20181012010 / OP_INDEX : 92 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20181012010 / OP_INDEX : 93 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20181012010 / OP_INDEX : 94 / SECTION : 0 -  데이터 선택 (4)
SHIP_ID : T20181012010 / OP_INDEX : 95 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20181012010 / OP_INDEX : 96 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20181012010 / OP_INDEX : 97 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20181012010 / OP_INDEX : 98 / SECTION : 0 -  데이터 선택 (5)
SHIP_ID : T20181012010 / OP_INDEX : 99 / SECTION : 0 -  데이터 선택 (9)
SHIP_ID : T20181012010 / OP_INDEX : 100 / SECTION : 0 -  데이터 선택 (9)
SHIP_ID : T20181012010 / OP_INDEX : 101 / SECTION : 0 -  데이터 선택 (7)
SHIP_ID : T20181012010 / OP_INDEX : 102 / SECTION : 0 -  데이터

SHIP_ID : T20190312005 / OP_INDEX : 232 / SECTION : 0 -  데이터 선택 (18)
SHIP_ID : T20190312005 / OP_INDEX : 233 / SECTION : 0 -  데이터 선택 (40)
SHIP_ID : T20190312005 / OP_INDEX : 234 / SECTION : 0 -  데이터 선택 (17)
SHIP_ID : T20190312005 / OP_INDEX : 235 / SECTION : 0 -  데이터 선택 (21)
SHIP_ID : T20190312005 / OP_INDEX : 236 / SECTION : 0 -  데이터 선택 (38)
SHIP_ID : T20190312005 / OP_INDEX : 237 / SECTION : 0 -  데이터 선택 (16)
SHIP_ID : T20190312005 / OP_INDEX : 238 / SECTION : 0 -  데이터 선택 (16)
SHIP_ID : T20190312005 / OP_INDEX : 239 / SECTION : 0 -  데이터 선택 (15)
SHIP_ID : T20190312005 / OP_INDEX : 240 / SECTION : 0 -  데이터 선택 (9)
SHIP_ID : T20190312005 / OP_INDEX : 241 / SECTION : 0 -  데이터 선택 (25)
SHIP_ID : T20190312005 / OP_INDEX : 242 / SECTION : 0 -  데이터 선택 (42)
SHIP_ID : T20190312005 / OP_INDEX : 243 / SECTION : 0 -  데이터 선택 (17)
SHIP_ID : T20190312005 / OP_INDEX : 244 / SECTION : 0 -  데이터 선택 (16)
SHIP_ID : T20190312005 / OP_INDEX : 245 / SECTION : 0 -  데이터 선택 (22)
SHIP_ID : T20190312005 / OP_INDEX :

...

optype : 1
distribute_by_application 함수 실행 시간: 4.1613초
전처리 후 학습 데이터 프레임이 존재합니다.
SHIP_ID : T20221031003 / OP_INDEX : 133 / SECTION : 0 -  데이터 선택 (173)
SHIP_ID : T20221031003 / OP_INDEX : 133 / SECTION : 0 -  조건 통과
optype : 1
distribute_by_application 함수 실행 시간: 2.0130초
전처리 후 학습 데이터 프레임이 존재합니다.
SHIP_ID : T20221031003 / OP_INDEX : 134 / SECTION : 0 -  데이터 선택 (298)
SHIP_ID : T20221031003 / OP_INDEX : 134 / SECTION : 0 -  조건 통과
optype : 1
distribute_by_application 함수 실행 시간: 2.0186초
전처리 후 학습 데이터 프레임이 존재합니다.
SHIP_ID : T20221031003 / OP_INDEX : 135 / SECTION : 0 -  데이터 선택 (298)
SHIP_ID : T20221031003 / OP_INDEX : 135 / SECTION : 0 -  조건 통과
optype : 1
distribute_by_application 함수 실행 시간: 2.2810초
전처리 후 학습 데이터 프레임이 존재합니다.
SHIP_ID : T20221031003 / OP_INDEX : 137 / SECTION : 0 -  데이터 선택 (113)
SHIP_ID : T20221031003 / OP_INDEX : 138 / SECTION : 0 -  데이터 선택 (100)
SHIP_ID : T20221031003 / OP_INDEX : 140 / SECTION : 0 -  데이터 선택 (21)
SHIP_ID : T20221031003 / OP_INDEX : 141 / SECTION : 0 -  데이터 선택 (20)
SHI

optype : 1
distribute_by_application 함수 실행 시간: 2.0293초
전처리 후 학습 데이터 프레임이 존재합니다.
SHIP_ID : T20221122010 / OP_INDEX : 1037 / SECTION : 0 -  데이터 선택 (6)
SHIP_ID : T20221122010 / OP_INDEX : 1038 / SECTION : 0 -  데이터 선택 (8)
SHIP_ID : T20221122010 / OP_INDEX : 1039 / SECTION : 0 -  데이터 선택 (174)
SHIP_ID : T20221122010 / OP_INDEX : 1039 / SECTION : 0 -  조건 통과
optype : 1
distribute_by_application 함수 실행 시간: 2.9505초
전처리 후 학습 데이터 프레임이 존재합니다.
SHIP_ID : T20221122010 / OP_INDEX : 1040 / SECTION : 0 -  데이터 선택 (92)
SHIP_ID : T20221122010 / OP_INDEX : 1041 / SECTION : 0 -  데이터 선택 (14)
SHIP_ID : T20221122010 / OP_INDEX : 1042 / SECTION : 0 -  데이터 선택 (10)
SHIP_ID : T20221122010 / OP_INDEX : 1043 / SECTION : 0 -  데이터 선택 (14)
SHIP_ID : T20221122010 / OP_INDEX : 1044 / SECTION : 0 -  데이터 선택 (12)
SHIP_ID : T20221122010 / OP_INDEX : 1045 / SECTION : 0 -  데이터 선택 (12)
SHIP_ID : T20221205001 / OP_INDEX : 83 / SECTION : 0 -  데이터 선택 (11)
SHIP_ID : T20221205001 / OP_INDEX : 84 / SECTION : 0 -  데이터 선택 (58)
SHIP_ID : T

SHIP_ID : T20230109001 / OP_INDEX : 697 / SECTION : 0 -  데이터 선택 (9)
SHIP_ID : T20230109002 / OP_INDEX : 182 / SECTION : 0 -  데이터 선택 (36)
SHIP_ID : T20230109002 / OP_INDEX : 183 / SECTION : 0 -  데이터 선택 (28)
SHIP_ID : T20230109002 / OP_INDEX : 184 / SECTION : 0 -  데이터 선택 (35)
SHIP_ID : T20230109002 / OP_INDEX : 185 / SECTION : 0 -  데이터 선택 (18)
SHIP_ID : T20230109002 / OP_INDEX : 186 / SECTION : 0 -  데이터 선택 (38)
SHIP_ID : T20230109002 / OP_INDEX : 187 / SECTION : 0 -  데이터 선택 (36)
SHIP_ID : T20230109002 / OP_INDEX : 188 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20230109002 / OP_INDEX : 189 / SECTION : 0 -  데이터 선택 (25)
SHIP_ID : T20230109002 / OP_INDEX : 190 / SECTION : 0 -  데이터 선택 (25)
SHIP_ID : T20230109002 / OP_INDEX : 191 / SECTION : 0 -  데이터 선택 (17)
SHIP_ID : T20230109002 / OP_INDEX : 192 / SECTION : 0 -  데이터 선택 (7)
SHIP_ID : T20230109002 / OP_INDEX : 193 / SECTION : 0 -  데이터 선택 (9)
SHIP_ID : T20230109002 / OP_INDEX : 194 / SECTION : 0 -  데이터 선택 (89)
SHIP_ID : T20230109002 / OP_INDEX : 19

SHIP_ID : T20230209001 / OP_INDEX : 797 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20230209001 / OP_INDEX : 798 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T20230209001 / OP_INDEX : 799 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T20230209001 / OP_INDEX : 800 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T20230209001 / OP_INDEX : 801 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20230209001 / OP_INDEX : 802 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20230209001 / OP_INDEX : 803 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T20230209001 / OP_INDEX : 804 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T20230209001 / OP_INDEX : 805 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T20230209001 / OP_INDEX : 807 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T20230209001 / OP_INDEX : 808 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T20230209001 / OP_INDEX : 809 / SECTION : 0 -  데이터 선택 (1)
SHIP_ID : T20230209001 / OP_INDEX : 810 / SECTION : 0 -  데이터 선택 (9)
SHIP_ID : T20230209001 / OP_INDEX : 811 / SECTION : 0 -  데이터 선택 (12)
SHIP_ID : T20230209001 / OP_INDEX : 813 / SECTI

SHIP_ID : T20230417001 / OP_INDEX : 265 / SECTION : 0 -  데이터 선택 (20)
SHIP_ID : T20230417001 / OP_INDEX : 266 / SECTION : 0 -  데이터 선택 (22)
SHIP_ID : T20230417001 / OP_INDEX : 267 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20230417001 / OP_INDEX : 268 / SECTION : 0 -  데이터 선택 (25)
SHIP_ID : T20230417001 / OP_INDEX : 269 / SECTION : 0 -  데이터 선택 (121)
SHIP_ID : T20230417001 / OP_INDEX : 271 / SECTION : 0 -  데이터 선택 (126)
SHIP_ID : T20230417001 / OP_INDEX : 273 / SECTION : 0 -  데이터 선택 (32)
SHIP_ID : T20230417001 / OP_INDEX : 274 / SECTION : 0 -  데이터 선택 (249)
SHIP_ID : T20230417001 / OP_INDEX : 274 / SECTION : 0 -  조건 통과
optype : 4
distribute_by_application 함수 실행 시간: 2.0606초
선박 데이터 프레임이 존재하지 않습니다.
SHIP_ID : T20230417001 / OP_INDEX : 275 / SECTION : 0 -  데이터 선택 (33)
SHIP_ID : T20230511002 / OP_INDEX : 209 / SECTION : 0 -  데이터 선택 (2)
SHIP_ID : T20230511002 / OP_INDEX : 210 / SECTION : 0 -  데이터 선택 (6)
SHIP_ID : T20230511002 / OP_INDEX : 211 / SECTION : 0 -  데이터 선택 (6)
SHIP_ID : T20230511002 / OP_INDE

In [7]:
# # 스케줄 설정: 3일에 한 번씩 데이터 가져오기
# #schedule.every(3).days.at("14:00").do(schedule_health_assessment)
# schedule.every(5).minutes.do(schedule_health_assessment)

# # 스케줄 지속 실행
# while True:
#     print("스케줄 시작")

#     schedule.run_pending()
#     time.sleep(1)